# Online Shoppers Purchasing Intention: Predicting Buyers vs. Browsers

**Problem.** Most online retail traffic is made of light visitors who never buy. Retailers cannot easily tell genuine buyers from casual browsers, so they either waste advertising budget or lose sales.

**Goal.** Build a supervised binary classifier on the UCI *Online Shoppers Purchasing Intention* dataset that predicts whether a web session ends in a purchase (`Revenue`).

# **1. Importing Libraries**

In [ ]:
import os, io, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_curve, roc_curve)

from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline   # resampling occurs only during fit, never during prediction

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# **2. Load the Dataset**

In [ ]:
CSV_NAME = "online_shoppers_intention.csv"

def load_data():
    for path in [CSV_NAME, f"/content/{CSV_NAME}", f"/mnt/user-data/uploads/{CSV_NAME}"]:
        if os.path.exists(path):
            return pd.read_csv(path)
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
            csv_inside = next(n for n in z.namelist() if n.lower().endswith(".csv"))
            with z.open(csv_inside) as f:
                return pd.read_csv(f)
    return pd.read_csv(io.BytesIO(uploaded[name]))

df = load_data()
print("Shape:", df.shape)
df.head()

# **3. Inspect the Data**

In [ ]:
df.info()

In [ ]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().sum() else "None")
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df.describe().T.round(3)

In [ ]:
print("Target distribution (raw):")
print(df["Revenue"].value_counts())
print((df["Revenue"].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

# **4. Cleaning Data**

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows -> {len(df)} rows remain")

df["Weekend"] = df["Weekend"].astype(int)
df["Revenue"] = df["Revenue"].astype(int)

print("\nClass balance after cleaning:")
print(df["Revenue"].value_counts())
print((df["Revenue"].value_counts(normalize=True) * 100).round(2).astype(str) + " %")

# **5. Exploratory Data Analysis**

In [ ]:
eda = df.copy()
eda["Outcome"] = eda["Revenue"].map({1: "Purchase", 0: "No purchase"})

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

sns.countplot(data=eda, x="Outcome", ax=axes[0], order=["No purchase", "Purchase"])
axes[0].set_title("Class balance")
for p in axes[0].patches:
    axes[0].annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                     ha="center", va="bottom")

sns.barplot(data=eda, x="VisitorType", y="Revenue", ax=axes[1], errorbar=None)
axes[1].set_title("Purchase rate by visitor type")
axes[1].set_ylabel("Purchase rate")

month_order = ["Feb", "Mar", "May", "June", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
sns.barplot(data=eda, x="Month", y="Revenue", order=month_order, ax=axes[2], errorbar=None)
axes[2].set_title("Purchase rate by month")
axes[2].set_ylabel("Purchase rate")

plt.tight_layout()
plt.show()

In [ ]:
features_to_plot = ["Administrative", "Informational", "ProductRelated",
                    "BounceRates", "ExitRates", "PageValues"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, feat in zip(axes.ravel(), features_to_plot):
    sns.boxplot(data=eda, x="Outcome", y=feat, ax=ax, order=["No purchase", "Purchase"])
    ax.set_title(f"{feat} by outcome")
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
num_cols_corr = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
                 "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates",
                 "PageValues", "SpecialDay", "Revenue"]
plt.figure(figsize=(10, 8))
sns.heatmap(df[num_cols_corr].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix (numerical features and target)")
plt.tight_layout()
plt.show()

**Analysis based on EDA**
- Only about 15-16% of sessions end in a purchase, so the data is imbalanced and accuracy alone would be misleading.
- `PageValues` separates buyers from non-buyers very clearly; `BounceRates` and `ExitRates` are higher for non-buyers.
- New visitors convert at a higher rate than returning visitors, and November has a much higher purchase rate than most other months.
- `BounceRates` and `ExitRates` are strongly correlated with each other, as are the page-count and duration columns. This is fine for Logistic Regression with regularisation, but worth knowing for the coefficient analysis.

# **6. Train / Test Split and Preprocessing Pipeline**

In [ ]:
X = df.drop(columns=["Revenue"])
y = df["Revenue"]

# Keep the final test set untouched until final evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

# Split the remaining 80% into model-fitting data (60% overall)
# and a separate threshold-validation set (20% overall).
X_train, X_threshold, y_train, y_threshold = train_test_split(
    X_train, y_train, test_size=0.25,
    random_state=RANDOM_STATE, stratify=y_train
)

print("Model-fitting set:", X_train.shape)
print("Threshold-validation set:", X_threshold.shape)
print("Final test set:", X_test.shape)
print("Purchase rates — fit: %.4f | threshold validation: %.4f | test: %.4f" %
      (y_train.mean(), y_threshold.mean(), y_test.mean()))

In [ ]:
numeric_cols = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
                "ProductRelated", "ProductRelated_Duration", "BounceRates", "ExitRates",
                "PageValues", "SpecialDay"]
categorical_cols = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
binary_cols = ["Weekend"]

def make_preprocessor(numeric=numeric_cols):
    # Fit scaling and one-hot encoding only on the current training fold.
    return ColumnTransformer([
        ("num", StandardScaler(), numeric),
        ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50,
                              sparse_output=False), categorical_cols),
        ("bin", "passthrough", binary_cols),
    ])

def make_pipeline(clf, use_smote=True, numeric=numeric_cols):
    # SMOTENC is applied to the original mixed-type columns. This avoids
    # creating fractional/invalid one-hot dummy variables as ordinary SMOTE can.
    steps = []
    if use_smote:
        steps.append(("smotenc", SMOTENC(
            categorical_features=categorical_cols + binary_cols,
            random_state=RANDOM_STATE
        )))
    steps.append(("prep", make_preprocessor(numeric)))
    steps.append(("clf", clf))
    return Pipeline(steps)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# **7. Evaluation Helper**

In [ ]:
def evaluate(name, y_true, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "Model": name,
        "PR-AUC": average_precision_score(y_true, prob),
        "ROC-AUC": roc_auc_score(y_true, prob),
        "Precision": precision_score(y_true, pred, zero_division=0),
        "Recall": recall_score(y_true, pred, zero_division=0),
        "F1": f1_score(y_true, pred, zero_division=0),
        "Accuracy": accuracy_score(y_true, pred),
    }

baseline_pr = y_test.mean()
print(f"No-skill PR-AUC baseline (purchase rate in test set): {baseline_pr:.3f}")

# **8. Baselines (no resampling)**


In [ ]:
results = []
fitted = {}

for name, clf in [("LR (baseline)", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
                  ("kNN k=5 (baseline)", KNeighborsClassifier(n_neighbors=5))]:
    pipe = make_pipeline(clf, use_smote=False)
    pipe.fit(X_train, y_train)
    fitted[name] = pipe
    results.append(evaluate(name, y_test, pipe.predict_proba(X_test)[:, 1]))

pd.DataFrame(results).round(4)

# **9. SMOTE + Logistic Regression and k-NN (tuned)**

Hyperparameters are tuned with 5-fold stratified cross-validation on the **training set only**.

In [ ]:
searches = {
    "LR + SMOTE": GridSearchCV(
        make_pipeline(LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
        param_grid={"clf__C": [0.01, 0.1, 1, 10, 100]},
        scoring="average_precision", cv=cv, n_jobs=-1),
    "kNN + SMOTE": GridSearchCV(
        make_pipeline(KNeighborsClassifier()),
        param_grid={"clf__n_neighbors": [3, 5, 11, 21, 31, 51],
                    "clf__weights": ["uniform", "distance"]},
        scoring="average_precision", cv=cv, n_jobs=-1),
}

for name, gs in searches.items():
    gs.fit(X_train, y_train)
    fitted[name] = gs.best_estimator_
    print(f"{name}: best params = {gs.best_params_} | CV PR-AUC = {gs.best_score_:.4f}")
    results.append(evaluate(name, y_test, gs.best_estimator_.predict_proba(X_test)[:, 1]))

# **10. Class-weighted Logistic Regression**

In [ ]:
cw_search = GridSearchCV(
    make_pipeline(LogisticRegression(max_iter=3000, class_weight="balanced",
                                     random_state=RANDOM_STATE), use_smote=False),
    param_grid={"clf__C": [0.01, 0.1, 1, 10, 100]},
    scoring="average_precision", cv=cv, n_jobs=-1)
cw_search.fit(X_train, y_train)

name = "LR + class weights"
fitted[name] = cw_search.best_estimator_
print(f"{name}: best params = {cw_search.best_params_} | CV PR-AUC = {cw_search.best_score_:.4f}")
results.append(evaluate(name, y_test, cw_search.best_estimator_.predict_proba(X_test)[:, 1]))

# **11. Ensembling: Soft Voting (Logistic Regression + k-NN)**

In [ ]:
# Reuse the best pipelines found using training-only CV.
from sklearn.ensemble import VotingClassifier


soft_vote = VotingClassifier(
    estimators=[("lr_smote", searches["LR + SMOTE"].best_estimator_), ("knn_smote", searches["kNN + SMOTE"].best_estimator_),], voting="soft", n_jobs=-1)

ensemble_search = GridSearchCV(estimator=soft_vote, param_grid={"weights": [(1, 1), (2, 1), (1, 2)]}, scoring="average_precision", cv=cv, n_jobs=-1, refit=True)
ensemble_search.fit(X_train, y_train)

ensemble_model = ensemble_search.best_estimator_
fitted["Soft Voting (LR + kNN)"] = ensemble_model

ensemble_test_prob = ensemble_model.predict_proba(X_test)[:, 1]
results.append(evaluate("Soft Voting (LR + kNN)", y_test, ensemble_test_prob))

print("Best voting weights:", ensemble_search.best_params_["weights"])
print("Cross-validated PR-AUC:", round(ensemble_search.best_score_, 4))
print("Held-out test metrics:")
display(pd.DataFrame([results[-1]]).set_index("Model").round(4))

# **12. Model Comparison**

In [ ]:
results_df = pd.DataFrame(results).set_index("Model")
display(results_df.round(4).sort_values("PR-AUC", ascending=False))
print(f"(No-skill PR-AUC = {baseline_pr:.3f})")

In [ ]:
plt.figure(figsize=(8, 6))
for name in ["LR (baseline)", "kNN k=5 (baseline)", "LR + SMOTE", "kNN + SMOTE", "LR + class weights"]:
    p, r, _ = precision_recall_curve(y_test, fitted[name].predict_proba(X_test)[:, 1])
    ap = average_precision_score(y_test, fitted[name].predict_proba(X_test)[:, 1])
    plt.plot(r, p, label=f"{name} (AP={ap:.3f})")
plt.axhline(baseline_pr, color="grey", linestyle="--", label=f"No skill ({baseline_pr:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curves on the test set")
plt.legend()
plt.show()

**Note** Precision, recall and F1 in the table above use a 0.5 cut-off. With resampling or class weights the model's probabilities are shifted, so 0.5 is rarely the best operating point. The next section chooses a threshold properly.

# **13. Logistic Regression/k-NN Baseline: Threshold Tuning**

The selected baseline is the SMOTE-based model with the higher training-only cross-validated PR-AUC. Its hyperparameters are selected using only the model-fitting set. The decision threshold is then selected on a separate stratified validation set that was not used to fit or tune the model. The final test set remains untouched until evaluation.

In [ ]:
candidates = {"LR + SMOTE": searches["LR + SMOTE"].best_score_,
              "kNN + SMOTE": searches["kNN + SMOTE"].best_score_}
final_name = max(candidates, key=candidates.get)
final_model = fitted[final_name]
print("Final model:", final_name, "| CV PR-AUC:", round(candidates[final_name], 4))

# Select the operating threshold on the separate validation set.
threshold_prob = final_model.predict_proba(X_threshold)[:, 1]
baseline_threshold_prob = threshold_prob.copy()

prec, rec, thr = precision_recall_curve(y_threshold, threshold_prob)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
best_threshold = float(thr[np.argmax(f1s)])
print(f"Validation threshold that maximises F1: {best_threshold:.3f} "
      f"(validation F1 = {f1s.max():.3f})")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(thr, prec[:-1], label="Precision")
plt.plot(thr, rec[:-1], label="Recall")
plt.plot(thr, f1s, label="F1", linewidth=2)
plt.axvline(best_threshold, color="grey", linestyle="--", label=f"Chosen threshold = {best_threshold:.2f}")
plt.xlabel("Decision threshold")
plt.title(f"Precision / recall / F1 vs. threshold ({final_name}, separate validation set)")
plt.legend()
plt.show()

# **14. Baseline Logistic Regression/k-NN Evaluation**

In [ ]:
test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= best_threshold).astype(int)

final_row = evaluate(f"{final_name} @ {best_threshold:.2f}", y_test, test_prob, best_threshold)
display(pd.DataFrame([final_row]).set_index("Model").round(4))

print(classification_report(y_test, test_pred, target_names=["No purchase", "Purchase"]))

ConfusionMatrixDisplay.from_predictions(y_test, test_pred,
                                        display_labels=["No purchase", "Purchase"], cmap="Blues")
plt.title("Confusion matrix on the test set")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

p, r, _ = precision_recall_curve(y_test, test_prob)
axes[0].plot(r, p, label=f"{final_name} (AP={average_precision_score(y_test, test_prob):.3f})")
axes[0].axhline(baseline_pr, color="grey", linestyle="--", label="No skill")
axes[0].scatter([recall_score(y_test, test_pred)], [precision_score(y_test, test_pred)],
                color="red", zorder=5, label="Chosen threshold")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-Recall curve"); axes[0].legend()

fpr, tpr, _ = roc_curve(y_test, test_prob)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, test_prob):.3f}")
axes[1].plot([0, 1], [0, 1], "--", color="grey")
axes[1].set_xlabel("False positive rate"); axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve"); axes[1].legend()

plt.tight_layout()
plt.show()

# **15. Baseline Interpretation: Logistic Regression Coefficients**

In [ ]:
lr_pipe = fitted["LR + SMOTE"]
feature_names = lr_pipe.named_steps["prep"].get_feature_names_out()
coefs = pd.DataFrame({"Feature": feature_names,
                      "Coefficient": lr_pipe.named_steps["clf"].coef_[0]})
coefs["Feature"] = coefs["Feature"].str.replace(r"^(num|cat|bin)__", "", regex=True)
coefs["abs"] = coefs["Coefficient"].abs()
top = coefs.sort_values("abs", ascending=False).head(12).sort_values("Coefficient")

plt.figure(figsize=(9, 6))
colors = ["tab:red" if c < 0 else "tab:green" for c in top["Coefficient"]]
plt.barh(top["Feature"], top["Coefficient"], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Coefficient (positive = more likely to purchase)")
plt.title("Top 12 Logistic Regression coefficients (LR + SMOTE)")
plt.tight_layout()
plt.show()

The numeric coefficients correspond to one-standard-deviation changes in their scaled features. One-hot coefficients correspond to a category indicator relative to the model's encoded reference/contrast structure. Their magnitudes should not be treated as directly comparable measures of feature importance. Correlated predictors can also make individual logistic-regression coefficients unstable.

# **16. Real-time Check**

`PageValues`, `BounceRates` and `ExitRates` are session-level statistics from web analytics. `PageValues` in particular is derived from pages that lead to a transaction, and these three features may only be fully known once a session ends. If so, they would not be available when the retailer needs to act.

To measure how much the model relies on them, retrain **without** these three features and compare.

In [ ]:
late_features = ["PageValues", "BounceRates", "ExitRates"]
early_numeric = [c for c in numeric_cols if c not in late_features]

ablation = GridSearchCV(
    make_pipeline(LogisticRegression(max_iter=3000, random_state=RANDOM_STATE), numeric=early_numeric),
    param_grid={"clf__C": [0.01, 0.1, 1, 10, 100]},
    scoring="average_precision", cv=cv, n_jobs=-1)
ablation.fit(X_train.drop(columns=late_features), y_train)

abl_prob = ablation.best_estimator_.predict_proba(X_test.drop(columns=late_features))[:, 1]

comparison = pd.DataFrame([
    evaluate("LR + SMOTE (all features)", y_test, fitted["LR + SMOTE"].predict_proba(X_test)[:, 1]),
    evaluate("LR + SMOTE (without PageValues, BounceRates, ExitRates)", y_test, abl_prob),
]).set_index("Model")
display(comparison.round(4))

# **17. Baseline Threshold: Business Interpretation**

In [ ]:
# Business interpretation on the separate threshold-validation set.
rows = []
for t in sorted(set([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, round(float(best_threshold), 2)])):
    pred = (baseline_threshold_prob >= t).astype(int)
    rows.append({"Threshold": t,
                 "Validation sessions flagged as likely buyers": int(pred.sum()),
                 "Precision": precision_score(y_threshold, pred, zero_division=0),
                 "Recall": recall_score(y_threshold, pred, zero_division=0)})
pd.DataFrame(rows).round(3).reset_index(drop=True)

The full-feature models use `PageValues`, `BounceRates` and `ExitRates`, which may not be available early enough for an intervention during a session. The early-feature experiment removes these fields. Verify feature availability timestamps before describing the early model as real-time.

The data is split into 60% model-fitting, 20% threshold-validation and 20% final test data. Hyperparameter selection uses stratified cross-validation on the model-fitting set. Decision thresholds are selected on the separate validation set. The test set is reserved for final reporting; avoid using its results to choose models or thresholds.

# **19. Tree-Based Model Experiments**

Random Forest, Extra Trees and HistGradientBoosting are compared on both feature sets. Hyperparameters are selected using stratified cross-validation on the model-fitting partition. Their validation results are exploratory; the final test set is reserved for the selected models.

In [ ]:

from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

# Separate full and early datasets
feature_sets = {
    "Full": (X_train, X_threshold, X_test, numeric_cols),
    "Early": (
        X_train.drop(columns=late_features),
        X_threshold.drop(columns=late_features),
        X_test.drop(columns=late_features),
        early_numeric
    )
}

model_configs = {
    "Random Forest": (
        RandomForestClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            "clf__max_depth": [None, 15],
            "clf__min_samples_leaf": [2, 5],
            "clf__max_features": ["sqrt"]
        }
    ),

    "Extra Trees": (
        ExtraTreesClassifier(
            n_estimators=300,
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        {
            "clf__max_depth": [None, 15],
            "clf__min_samples_leaf": [2, 5],
            "clf__max_features": ["sqrt"]
        }
    ),

    "HistGradientBoosting": (
        HistGradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
        {
            "clf__max_iter": [100, 200],
            "clf__learning_rate": [0.05, 0.1],
            "clf__max_leaf_nodes": [15, 31]
        }
    )
}

experiment_results = []
experiment_models = {}

for scope, (Xtr, Xval, Xte, num_cols) in feature_sets.items():

    print(f"\n========== {scope} MODEL ==========")

    for model_name, (clf, params) in model_configs.items():

        pipe = make_pipeline(clf, numeric=num_cols)

        search = GridSearchCV(
            pipe,
            param_grid=params,
            scoring="average_precision",
            cv=cv,
            n_jobs=-1,
            refit=True
        )

        search.fit(Xtr, y_train)

        best_model = search.best_estimator_

        # Select threshold on the separate validation set; it is not used
        # for fitting or hyperparameter selection.
        val_prob = best_model.predict_proba(Xval)[:, 1]
        p, r, thresholds = precision_recall_curve(y_threshold, val_prob)

        f1_values = (
            2 * p[:-1] * r[:-1] /
            (p[:-1] + r[:-1] + 1e-12)
        )

        best_t = float(thresholds[np.argmax(f1_values)])

        # Evaluate on held-out test set
        test_prob = best_model.predict_proba(Xte)[:, 1]

        row = evaluate(
            f"{scope} - {model_name}",
            y_test,
            test_prob,
            best_t
        )

        row["CV PR-AUC"] = search.best_score_
        row["Threshold"] = best_t

        experiment_results.append(row)

        experiment_models[f"{scope} - {model_name}"] = {
            "model": best_model,
            "threshold": best_t,
            "params": search.best_params_,
            "cv_pr_auc": float(search.best_score_)
        }

        print(
            model_name,
            "| CV PR-AUC:", round(search.best_score_, 4),
            "| Threshold:", round(best_t, 3),
            "| Test F1:", round(row["F1"], 4)
        )

experiment_df = pd.DataFrame(experiment_results)

display(
    experiment_df.sort_values(
        "CV PR-AUC", ascending=False
    ).set_index("Model").round(4)
)



For each feature set, compare two imbalance strategies:

- **SMOTENC:** resample the original mixed numeric/categorical training-fold data while preserving categorical levels, then scale/one-hot encode.
- **Class weighting:** use `scale_pos_weight` without oversampling.

Randomized search selects hyperparameters using stratified cross-validation on the model-fitting set. Decision thresholds are selected on the separate validation set, not on the final test set.

In [ ]:
# XGBoost is used as the boosted-tree model for this comparison.
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

# Prepare both feature sets
feature_sets = {
    "Full": (X_train, X_threshold, X_test, numeric_cols),
    "Early": (
        X_train.drop(columns=late_features),
        X_threshold.drop(columns=late_features),
        X_test.drop(columns=late_features),
        early_numeric
    )
}

xgb_results = []
xgb_models = {}

# Search space
base_params = {
    "clf__n_estimators": randint(150, 501),
    "clf__max_depth": randint(2, 8),
    "clf__learning_rate": loguniform(0.02, 0.2),
    "clf__min_child_weight": randint(1, 10),
    "clf__subsample": uniform(0.65, 0.35),
    "clf__colsample_bytree": uniform(0.65, 0.35),
    "clf__reg_lambda": loguniform(0.1, 20),
    "clf__gamma": [0, 0.1, 0.3, 0.5]
}

for scope, (Xtr, Xval, Xte, num_cols) in feature_sets.items():

    for strategy in ["SMOTE", "Class Weight"]:

        print(f"\n{scope} MODEL - {strategy}")

        if strategy == "SMOTE":
            param_dist = [
                {
                    **base_params,
                    "clf__scale_pos_weight": [1]
                }
            ]
            use_smote = True

        else:
            param_dist = [
                {
                    **base_params,
                    "clf__scale_pos_weight": [1, 2, 3, 4]
                }
            ]
            use_smote = False

        xgb = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=1
        )

        pipe = make_pipeline(
            xgb,
            use_smote=use_smote,
            numeric=num_cols
        )

        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=param_dist,
            n_iter=20,
            scoring="average_precision",
            cv=cv,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            refit=True
        )

        search.fit(Xtr, y_train)

        best_model = search.best_estimator_

        # Select threshold on the separate validation set.
        val_prob = best_model.predict_proba(Xval)[:, 1]
        p, r, thresholds = precision_recall_curve(y_threshold, val_prob)

        f1_values = (
            2 * p[:-1] * r[:-1] /
            (p[:-1] + r[:-1] + 1e-12)
        )

        best_t = float(thresholds[np.argmax(f1_values)])

        # Test evaluation
        test_prob = best_model.predict_proba(Xte)[:, 1]

        result = evaluate(
            f"{scope} XGBoost - {strategy}",
            y_test,
            test_prob,
            best_t
        )

        result["CV PR-AUC"] = search.best_score_
        result["Threshold"] = best_t

        xgb_results.append(result)

        key = f"{scope} - XGBoost - {strategy}"

        xgb_models[key] = {
            "model": best_model,
            "threshold": best_t,
            "params": search.best_params_,
            "cv_pr_auc": float(search.best_score_)
        }

        print("Best parameters:", search.best_params_)
        print("CV PR-AUC:", round(search.best_score_, 4))
        print("Threshold:", round(best_t, 4))
        print("Test F1:", round(result["F1"], 4))

xgb_df = pd.DataFrame(xgb_results)

display(
    xgb_df.sort_values(
        "CV PR-AUC", ascending=False
    ).set_index("Model").round(4)
)


# **21. Final Candidate Comparison, Evaluation and Model Export**

The full-feature and early-feature XGBoost candidates are selected by training-only cross-validated PR-AUC. The final test set is evaluated only here, after the model-selection and threshold-selection steps are complete.

In [ ]:
# Select the highest training-CV PR-AUC candidate separately for each feature set.
# The final test set is not used for this selection.
full_key = max(
    (key for key in xgb_models if key.startswith("Full - ")),
    key=lambda key: xgb_models[key]["cv_pr_auc"]
)
early_key = max(
    (key for key in xgb_models if key.startswith("Early - ")),
    key=lambda key: xgb_models[key]["cv_pr_auc"]
)

print("Selected full-feature candidate:", full_key,
      "| CV PR-AUC:", round(xgb_models[full_key]["cv_pr_auc"], 4))
print("Selected early-feature candidate:", early_key,
      "| CV PR-AUC:", round(xgb_models[early_key]["cv_pr_auc"], 4))

full_model = xgb_models[full_key]["model"]
full_threshold = float(xgb_models[full_key]["threshold"])
early_model = xgb_models[early_key]["model"]
early_threshold = float(xgb_models[early_key]["threshold"])

# Evaluate only the final selected models on the untouched test set.
X_test_early = X_test.drop(columns=late_features)
full_prob = full_model.predict_proba(X_test)[:, 1]
early_prob = early_model.predict_proba(X_test_early)[:, 1]
full_pred = (full_prob >= full_threshold).astype(int)
early_pred = (early_prob >= early_threshold).astype(int)

final_comparison = pd.DataFrame([
    evaluate(f"Full — {full_key.split(' - ', 1)[1]}", y_test, full_prob, full_threshold),
    evaluate(f"Early — {early_key.split(' - ', 1)[1]}", y_test, early_prob, early_threshold),
]).set_index("Model")
final_comparison["Threshold"] = [full_threshold, early_threshold]
display(final_comparison.round(4))

print("FULL MODEL CLASSIFICATION REPORT")
print(classification_report(y_test, full_pred,
                            target_names=["No purchase", "Purchase"], zero_division=0))
print("EARLY MODEL CLASSIFICATION REPORT")
print(classification_report(y_test, early_pred,
                            target_names=["No purchase", "Purchase"], zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, full_pred, display_labels=["No purchase", "Purchase"],
    cmap="Blues", ax=axes[0], colorbar=False
)
axes[0].set_title(f"Full model (threshold={full_threshold:.3f})")
ConfusionMatrixDisplay.from_predictions(
    y_test, early_pred, display_labels=["No purchase", "Purchase"],
    cmap="Greens", ax=axes[1], colorbar=False
)
axes[1].set_title(f"Early model (threshold={early_threshold:.3f})")
plt.tight_layout()
plt.show()

import joblib
bundle = {
    "columns": list(X_train.columns),
    "full": {
        "model": full_model, "threshold": full_threshold,
        "name": full_key.split(" - ", 1)[1],
        "features": list(X_train.columns),
    },
    "early": {
        "model": early_model, "threshold": early_threshold,
        "name": early_key.split(" - ", 1)[1],
        "dropped_features": list(late_features),
        "features": list(X_train.drop(columns=late_features).columns),
    },
    "numeric_ranges": {
        c: (float(X_train[c].min()), float(X_train[c].quantile(0.995)))
        for c in numeric_cols
    },
    "numeric_defaults": X_train[numeric_cols].median(numeric_only=True).to_dict(),
    "categorical_values": {
        c: sorted(X_train[c].dropna().astype(str).unique().tolist())
        for c in categorical_cols
    },
    "random_state": RANDOM_STATE,
    "split_proportions": {"fit": 0.60, "threshold_validation": 0.20, "test": 0.20},
    "threshold_selection": "separate stratified validation set",
    "resampling": "SMOTENC before scaling and one-hot encoding",
}
joblib.dump(bundle, "shopper_model_bundle.joblib")
print("Saved: shopper_model_bundle.joblib")
print(f"Full threshold: {full_threshold:.4f}")
print(f"Early threshold: {early_threshold:.4f}")

# **22. Conclusion and Limitations**

- The target is imbalanced (approximately 84% no-purchase and 16% purchase), so PR-AUC, precision, recall and F1 should accompany accuracy.
- The notebook uses a stratified 60% model-fitting / 20% threshold-validation / 20% final-test split. Hyperparameters are selected using five-fold stratified CV on the model-fitting set.
- SMOTENC is applied to the original mixed-type predictors inside the training pipeline, before scaling and one-hot encoding. This avoids synthetic fractional one-hot indicators and prevents resampling the validation or test data.
- Decision thresholds are selected using the separate validation set. The final test set is used only for the final reported evaluation; do not use those results to choose a model or threshold.
- Full-feature models can use session-level signals such as `PageValues`, `BounceRates` and `ExitRates`. Confirm when each field becomes available before claiming the model can intervene during a live session.
- The early-feature model excludes those three fields, but this alone does not prove every remaining predictor is available early enough for real-time use.
- The saved model bundle contains both fitted pipelines, their thresholds, feature metadata and split details. The Streamlit application must use the same input column names, types, feature order and selected threshold.
- The final test metrics should be reported once, after model selection is complete. Any further model experimentation should use a new holdout set or nested validation.